In [6]:
import pandas as pd
import numpy as np
import duckdb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import ipaddress
from collections import Counter
import warnings
warnings.filterwarnings('ignore')
import json

# Display options
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 100)
pd.set_option('display.width', 200)

# File path — update for your machine
DATA_PATH = Path(r"C:\Users\dunca\OneDrive\Documents\UON\SCS6104\Capstone\data\raw\botsv1.stream_http.csv")
print(f"File exists: {DATA_PATH.exists()}, Size: {DATA_PATH.stat().st_size / 1e9:.2f} GB")

File exists: True, Size: 0.30 GB


In [2]:
# Read just the header + first 5 rows as text
with open(DATA_PATH, 'r', encoding='utf-8', errors='replace') as f:
    for i, line in enumerate(f):
        if i < 6:
            print(f"--- Line {i} ---")
            print(line[:500])  # truncate long lines
        else:
            break

--- Line 0 ---
"_serial","_time",source,sourcetype,host,index,"splunk_server","_raw"

--- Line 1 ---
0,"2016-08-28 17:54:58.348 MDT","stream:http","stream:http","splunk-02",botsv1,"kraken.local","{""endtime"":""2016-08-28T23:54:58.348077Z"",""timestamp"":""2016-08-28T23:54:58.240395Z"",""ack_packets_in"":5,""ack_packets_out"":4,""bytes"":4900,""bytes_in"":213,""bytes_out"":4687,""c_ip"":""192.168.250.100"",""cached"":0,""capture_hostname"":""demo-01"",""client_rtt"":123,""client_rtt_packets"":2,""client_rtt_sum"":247,""connection_type"":""Keep-Alive"",""cs_version"":[""1.1"",""1.1""],""data_cen
--- Line 2 ---
"

--- Line 3 ---
1,"2016-08-28 17:51:26.774 MDT","stream:http","stream:http","splunk-02",botsv1,"kraken.local","{""endtime"":""2016-08-28T23:51:26.774611Z"",""timestamp"":""2016-08-28T23:51:26.586434Z"",""ack_packets_in"":3,""ack_packets_out"":8,""bytes"":2162,""bytes_in"":1984,""bytes_out"":178,""c_ip"":""192.168.250.100"",""cached"":0,""capture_hostname"":""demo-01"",""client_r

In [3]:
# Fast row count using PowerShell-style counting via Python
def count_rows_fast(filepath, chunk_size=1024*1024):
    """Count lines without loading file into memory."""
    count = 0
    with open(filepath, 'rb') as f:
        while chunk := f.read(chunk_size):
            count += chunk.count(b'\n')
    return count

total_rows = count_rows_fast(DATA_PATH)
print(f"Total rows (incl. header): {total_rows:,}")

# Get column count from header
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    header = f.readline().strip().split(',')
print(f"Total columns: {len(header)}")
print(f"First 20 columns: {header[:20]}")

Total rows (incl. header): 54,085
Total columns: 8
First 20 columns: ['"_serial"', '"_time"', 'source', 'sourcetype', 'host', 'index', '"splunk_server"', '"_raw"']


In [4]:
# Connect to in-memory DuckDB
con = duckdb.connect()

# Profile the file lazily — DuckDB reads only what's needed
schema_query = f"""
DESCRIBE
SELECT * FROM read_csv_auto('{DATA_PATH.as_posix()}', sample_size=10000)
"""
schema_df = con.execute(schema_query).fetchdf()
print(f"Detected {len(schema_df)} columns")
schema_df.head(50)

Detected 8 columns


,column_name,column_type,null,key,default,extra
0,_serial,BIGINT,YES,None,None,None
1,_time,VARCHAR,YES,None,None,None
2,source,VARCHAR,YES,None,None,None
3,sourcetype,VARCHAR,YES,None,None,None
4,host,VARCHAR,YES,None,None,None
5,index,VARCHAR,YES,None,None,None
6,splunk_server,VARCHAR,YES,None,None,None
7,_raw,VARCHAR,YES,None,None,None


In [7]:


# Read just 5 rows
sample = pd.read_csv(DATA_PATH, nrows=5)
print(sample.columns.tolist())
print(f"Shape: {sample.shape}")

# Parse the _raw JSON for the first row and see ALL its keys
first_raw = sample['_raw'].iloc[0]
parsed = json.loads(first_raw)
print(f"\nNumber of fields inside _raw JSON: {len(parsed)}")
print(f"\nAll JSON field names:")
for key in sorted(parsed.keys()):
    val = parsed[key]
    val_preview = str(val)[:80]
    print(f"  {key:35s} = {val_preview}")

['_serial', '_time', 'source', 'sourcetype', 'host', 'index', 'splunk_server', '_raw']
Shape: (5, 8)

Number of fields inside _raw JSON: 59

All JSON field names:
  ack_packets_in                      = 5
  ack_packets_out                     = 4
  bytes                               = 4900
  bytes_in                            = 213
  bytes_out                           = 4687
  c_ip                                = 192.168.250.100
  cached                              = 0
  capture_hostname                    = demo-01
  client_rtt                          = 123
  client_rtt_packets                  = 2
  client_rtt_sum                      = 247
  connection_type                     = Keep-Alive
  cs_version                          = ['1.1', '1.1']
  data_center_time                    = 107682
  data_packets_in                     = 1
  data_packets_out                    = 4
  dest_content                        = <?xml version="1.0" encoding="utf-8"?><tile><visual version="2" Br

In [8]:
# Check across 1000 records to handle field sparsity
sample_1k = pd.read_csv(DATA_PATH, nrows=1000)
all_keys = set()
for raw in sample_1k['_raw']:
    try:
        parsed = json.loads(raw)
        all_keys.update(parsed.keys())
    except (json.JSONDecodeError, TypeError):
        continue

print(f"Total unique JSON fields across 1000 records: {len(all_keys)}")
print(f"\nFields sorted alphabetically:")
for k in sorted(all_keys):
    print(f"  {k}")

Total unique JSON fields across 1000 records: 74

Fields sorted alphabetically:
  accept
  accept_language
  ack_packets_in
  ack_packets_out
  bytes
  bytes_in
  bytes_out
  c_ip
  cached
  canceled
  capture_hostname
  client_rtt
  client_rtt_packets
  client_rtt_sum
  connection_type
  content_encoding
  cs_cache_control
  cs_content_length
  cs_content_type
  cs_date
  cs_pragma
  cs_version
  data_center_time
  data_packets_in
  data_packets_out
  dest_content
  dest_headers
  dest_ip
  dest_mac
  dest_port
  duplicate_packets_in
  duplicate_packets_out
  endtime
  etag
  expires
  form_data
  http_comment
  http_content_length
  http_content_type
  http_method
  http_referrer
  http_user_agent
  missing_packets_in
  missing_packets_out
  network_interface
  packets_in
  packets_out
  reply_time
  request
  request_ack_time
  request_time
  response_ack_time
  response_time
  sc_cache_control
  sc_date
  server
  server_rtt
  server_rtt_packets
  server_rtt_sum
  set_cookie
  site

In [14]:
# Check the date range — attack-only should span just attack windows
print(con.execute("""SELECT MIN(event_time) AS earliest, MAX(event_time) AS latest, COUNT(DISTINCT DATE_TRUNC('day', event_time)) AS distinct_days FROM raw_stream_http """).fetchdf())

BinderException: Binder Error: Referenced column "event_time" not found in FROM clause!
Candidate bindings: "raw_stream_http.sourcetype", "raw_stream_http.splunk_server"
LINE 1: SELECT MIN(event_time) AS earliest, MAX(event_time...
                   ^